# E45 Account A — prepare, exact 2×T4 DDP train, and candidate arm

No public/private generation is performed. The 200-row answer-blind holdout is local experiment control only.


In [ ]:
# Runtime pin and dual-T4 preflight.  Do not replace torch: Kaggle must expose
# the frozen 2.10.0+cu128 build, otherwise this experiment stops.
import importlib.metadata, subprocess, sys

PIP_PINS = {
    "transformers": "5.16.1",
    "peft": "0.19.1",
    "accelerate": "1.13.0",
    "bitsandbytes": "0.50.2",
    "sentence-transformers": "5.4.1",
    "faiss-cpu": "1.15.0",
    "numpy": "2.0.2",
}
installed = {dist.metadata["Name"].lower(): dist.version for dist in importlib.metadata.distributions()}
need = [f"{name}=={version}" for name, version in PIP_PINS.items() if installed.get(name) != version]
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", *need], check=True)

# Kaggle currently exposes torchao 0.10.0. Transformers 5.16 rejects that
# optional version during Qwen model loading, while this FP16/NF4 experiment
# does not use torchao at all. Make its absence an explicit preflight invariant.
try:
    importlib.metadata.version("torchao")
except importlib.metadata.PackageNotFoundError:
    pass
else:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
try:
    importlib.metadata.version("torchao")
except importlib.metadata.PackageNotFoundError:
    pass
else:
    raise AssertionError("torchao must be absent for the frozen E45 runtime")

import faiss
import numpy
import sentence_transformers
import torch
actual = {name: importlib.metadata.version(name) for name in PIP_PINS}
assert actual == PIP_PINS, (actual, PIP_PINS)
assert hasattr(faiss, "read_index") and hasattr(faiss, "IndexFlatIP"), "FAISS import lacks required index APIs"
assert numpy.__version__ == PIP_PINS["numpy"]
assert sentence_transformers.__version__ == PIP_PINS["sentence-transformers"]
assert str(torch.__version__) == "2.10.0+cu128", torch.__version__
assert torch.cuda.is_available() and torch.cuda.device_count() == 2, (torch.cuda.is_available(), torch.cuda.device_count())
print({"runtime": {**actual, "torch": torch.__version__}, "gpus": [torch.cuda.get_device_name(i) for i in range(2)]})


In [ ]:
# Resolve one system archive by SHA, verify its sidecar, then safely extract the
# direct src/configs layout.  This code deliberately refuses nested roots.
import hashlib, json, os, shutil, stat, sys, zipfile
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
PROJECT_ROOT = WORK / "e45-system"
EXPECTED_SYSTEM_SHA256 = "bc02b22158a4154c6d39e92cd743d5e296352c1f66ba7ed7e03fc197ab6531c0"

def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

archives = [path for path in INPUT_ROOT.rglob("*.bin") if sha256(path) == EXPECTED_SYSTEM_SHA256]
assert len(archives) == 1, f"expected exactly one E45 R3 system archive, found {archives}"
system_bin = archives[0]
sidecar = Path(str(system_bin) + ".sha256")
assert sidecar.is_file() and sidecar.read_text(encoding="utf-8").split()[0].lower() == EXPECTED_SYSTEM_SHA256
assert not PROJECT_ROOT.exists(), f"refusing to reuse extraction directory {PROJECT_ROOT}"
PROJECT_ROOT.mkdir(parents=True)
with zipfile.ZipFile(system_bin) as archive:
    declared_manifest = json.loads(archive.read("SYSTEM_FILE_MANIFEST.json").decode("utf-8"))
    actual_members = {info.filename for info in archive.infolist()}
    assert actual_members == set(declared_manifest) | {"SYSTEM_FILE_MANIFEST.json"}, "system member manifest mismatch"
    seen = set()
    for info in archive.infolist():
        name = info.filename.replace("\\", "/")
        if name == "SYSTEM_FILE_MANIFEST.json":
            continue
        assert name and not name.endswith("/")
        assert not name.startswith("/") and ".." not in name.split("/") and name.casefold() not in seen
        assert not stat.S_ISLNK(info.external_attr >> 16), f"symlink member: {name}"
        payload = archive.read(info)
        assert len(payload) == declared_manifest[name]["bytes"]
        assert hashlib.sha256(payload).hexdigest() == declared_manifest[name]["sha256"]
        seen.add(name.casefold())
        target = (PROJECT_ROOT / name).resolve()
        assert target.is_relative_to(PROJECT_ROOT.resolve())
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(payload)
assert (PROJECT_ROOT / "src/uit_dsc_fixed_rag/__init__.py").is_file()
assert (PROJECT_ROOT / "configs/e45-inference-aligned-parent-lora-v1.json").is_file()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print({"system": str(system_bin), "sha256": EXPECTED_SYSTEM_SHA256, "project": str(PROJECT_ROOT)})


In [ ]:
from uit_dsc_fixed_rag.e45_input_resolver import build_verified_artifact_directory, resolve_required_artifacts

COMMON = {
    "official_train": "2a52501cc065d266f2f832475950bcf1e7c75c386efa9b2f568f251d745f5988",
    "official_warmup": "b824e4f18bd9181c021498a28e402b7374d6d559f2ff28caa4120a9d932f82c5",
    "official_public": "5f68ca901cb20798559538bef60fa7c32bd7d0df59f5bf31a37eb220c9e00df5",
    "splits_train": "53db63c15f779babe99c3fcfcf4f4ecf864a7eb0e738637b9e717e1fe357d921",
    "splits_dev": "2209b5e066ee354aae00f3b2b1aa69dbf71097c2ccd354f18b2f60daab128dcb",
}
E00 = {
    "manifest.json": "04efd3905ad6d2758461587ca68d7d70fa2c568855b78f0c762c7a37ba547b2e",
    "bm25.sqlite3": "f874a9528433f0db64efe7b3e951028d89433cbfb4c6e951a182b531acf1e0f0",
    "chunks.jsonl": "1e48c7762765ac2dd169045e9f5327c5311db3f1da8a6007ef70fff58718e367",
    "documents.jsonl": "f2968724e8a25124034b9ff2144427f8853ed37359443bd149ec3832f4c1fed7",
}
E02 = {
    "manifest.json": "1b4d23c0149457055559ba1327b4dd5a1ae60bf07fa15a70b1e41d4ea514b270",
    "dense.faiss": "96ab6b8afcb376e327116642e0ffc378d633d9f712b698afd0e43dcee654a633",
    "chunk_ids.jsonl": "6e26962a0963f50460ada74707db31e604dfe4991e7d7150afeec89aa363fb99",
}
resolved = resolve_required_artifacts([INPUT_ROOT], COMMON)
e00_dir = build_verified_artifact_directory(search_roots=[INPUT_ROOT], destination=WORK / "verified-e00", required_files=E00)
e02_dir = build_verified_artifact_directory(search_roots=[INPUT_ROOT], destination=WORK / "verified-e02", required_files=E02)
print({key: str(value) for key, value in resolved.items()})

EXTRA_A = {"e08a_results": "8bb590020510bc512dbcf738165480cd164059826849a001d117c2374e8e0786", "prior_records": "1d7c5b97e539cfe33ce036a66de4595c959d30b0a39b10e10c2fba415ce3d46b"}
resolved.update(resolve_required_artifacts([INPUT_ROOT], EXTRA_A))


In [ ]:
# Optional recovery from one attached prior failed account output. The shared
# preparation function validates all 200 records and every bound hash before
# accepting this file; an invalid or ambiguous candidate fails closed.
prior_context_candidates = list(INPUT_ROOT.rglob("holdout-contexts/holdout-contexts.jsonl"))
assert len(prior_context_candidates) <= 1, f"multiple prior holdout artifacts: {prior_context_candidates}"
if prior_context_candidates:
    recovery_dir = WORK / "holdout-contexts"
    recovery_target = recovery_dir / "holdout-contexts.jsonl"
    if not recovery_target.is_file():
        recovery_dir.mkdir(parents=True, exist_ok=False)
        shutil.copy2(prior_context_candidates[0], recovery_target)
        print({"recovery_candidate": str(prior_context_candidates[0])})

import json, logging
from transformers import AutoTokenizer
from uit_dsc_fixed_rag.e45_holdout_contexts import prepare_holdout_contexts, select_group_safe_holdout
from uit_dsc_fixed_rag.e45_parent_training import load_config

config = load_config(PROJECT_ROOT / "configs/e45-inference-aligned-parent-lora-v1.json")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s", force=True)
print("[E45] Cell 4/8: selecting holdout, validating FAISS, then preparing 200 contexts...", flush=True)
questions, split_identity = select_group_safe_holdout(
    official_train_path=resolved["official_train"], official_warmup_path=resolved["official_warmup"],
    official_public_path=resolved["official_public"], splits_train_path=resolved["splits_train"],
    splits_dev_path=resolved["splits_dev"], config=config,
)
assert split_identity["sample_size"] == 200 and split_identity["group_count"] == 199
tokenizer_dir = WORK / "frozen-viqwen-tokenizer"
tokenizer = AutoTokenizer.from_pretrained("AITeamVN/Vi-Qwen2-3B-RAG", revision=config.section("model_inventory")["base_revision"], trust_remote_code=False, fix_mistral_conversions=False)
tokenizer.save_pretrained(tokenizer_dir)
contexts_path, contexts_manifest = prepare_holdout_contexts(
    questions=questions, e00_dir=e00_dir, dense_dir=e02_dir, tokenizer_path=tokenizer_dir,
    output_dir=WORK / "holdout-contexts", config=config, device="cuda:0",
)
print({"split": split_identity, "contexts": contexts_manifest})


In [ ]:
# Materialize the real 5,636 rows and require all computed static gates before model loading.
import subprocess
prep_dir = WORK / "e45-preparation"
prior_materializations = list(INPUT_ROOT.rglob("e45-preparation/training-records.jsonl"))
assert len(prior_materializations) <= 1, f"multiple prior E45 materializations: {prior_materializations}"
reuse_args = []
if prior_materializations:
    prep_dir.mkdir(parents=True, exist_ok=False)
    recovered_records = prep_dir / "training-records.jsonl"
    shutil.copy2(prior_materializations[0], recovered_records)
    assert sha256(recovered_records) == "84fb204f6268ebd4b0a56f1e4c5048c31a4e91778b9ac3c261d9de509a5ca17f", "prior materialization identity mismatch"
    reuse_args = ["--skip-materialization-if-exists"]
    print({"recovered_materialization": str(prior_materializations[0]), "sha256": sha256(recovered_records)})
subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts/run_e45_static_prepare.py"),
    "--config-path", str(PROJECT_ROOT / "configs/e45-inference-aligned-parent-lora-v1.json"),
    "--train-json", str(resolved["official_train"]), "--e00-dir", str(e00_dir),
    "--e08a-results", str(resolved["e08a_results"]), "--prior-records", str(resolved["prior_records"]),
    "--tokenizer-path", str(tokenizer_dir), "--output-dir", str(prep_dir), *reuse_args], check=True)
prep_manifest = json.loads((prep_dir / "preparation-manifest.json").read_text(encoding="utf-8"))
assert prep_manifest["all_gates_pass"] is True, prep_manifest["gate_checks"]


In [ ]:
# Optional cross-session resume: zero or exactly one sidecar-verified archive is allowed.
import subprocess
from uit_dsc_fixed_rag.e45_input_resolver import resolve_and_extract_single_resume_checkpoint
train_dir = WORK / "e45-training"
resume = resolve_and_extract_single_resume_checkpoint(search_roots=[INPUT_ROOT], destination_parent=WORK / "resume-extract")
command = ["torchrun", "--nproc_per_node=2", str(PROJECT_ROOT / "scripts/run_e45_train_kaggle.py"),
    "--config-path", str(PROJECT_ROOT / "configs/e45-inference-aligned-parent-lora-v1.json"),
    "--records-jsonl", str(prep_dir / "training-records.jsonl"), "--manifest-json", str(prep_dir / "preparation-manifest.json"),
    "--base-model-path", "AITeamVN/Vi-Qwen2-3B-RAG", "--tokenizer-path", str(tokenizer_dir),
    "--output-dir", str(train_dir), "--max-runtime-hours", "11.25"]
if resume is not None:
    command += ["--resume-checkpoint-dir", str(resume), "--skip-smoke"]
    print({"resume_policy": "validated checkpoint", "skip_smoke": True, "reason": "the 697-step segment already passed smoke; 8 optimizer steps remain"}, flush=True)
    assert "--skip-smoke" in command, "FINAL-V2 resume must skip the already-passed full-run smoke"
print({"notebook_build": "E45_ACCOUNT_A_RESUME_STEP697_FINAL_V2", "torchrun_command": command}, flush=True)
# PEFT 0.19.1 imports a legacy Transformers TP symbol while loading a LoRA
# checkpoint. Transformers 5.16.1 removed only that compatibility export.
# This DDP run has no tensor-parallel plan, so the symbol is imported but never
# used. Supply the legacy alias outside PROJECT_ROOT to preserve code identity.
shim_dir = WORK / "e45-resume-runtime-shim"
shim_dir.mkdir(parents=True, exist_ok=True)
(shim_dir / "sitecustomize.py").write_text(
    "import transformers.integrations.tensor_parallel as _tp\n"
    "if not hasattr(_tp, 'EmbeddingParallel'):\n"
    "    _tp.EmbeddingParallel = _tp.ColwiseParallel\n",
    encoding="utf-8",
)
train_environment = os.environ.copy()
train_environment["PYTHONPATH"] = str(shim_dir) + os.pathsep + train_environment.get("PYTHONPATH", "")
train_environment["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
subprocess.run(command, check=True, env=train_environment)
complete = train_dir / "complete.json"
if not complete.is_file():
    checkpoints = sorted(train_dir.glob("E45_ACCOUNT_A_CHECKPOINT_STEP_*.bin"))
    assert len(checkpoints) == 1, "training did not complete and did not produce one resumable checkpoint"
    raise RuntimeError(f"Session stopped safely at {checkpoints[0].name}; attach it to the next Account A session.")


In [ ]:
# Direct-private path authorized by the operator: package the certified 705-step
# adapter without generating or scoring the 200-row holdout.
import hashlib, json, shutil, zipfile
from pathlib import Path

def _file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

adapter_dir = train_dir / "adapter-final"
complete_path = train_dir / "complete.json"
assert complete_path.is_file(), "Training did not produce complete.json"
complete = json.loads(complete_path.read_text(encoding="utf-8"))
assert complete["global_step"] == complete["expected_total_steps"] == 705, complete
assert (adapter_dir / "adapter_model.safetensors").is_file()
assert (adapter_dir / "adapter_config.json").is_file()
assert _file_sha256(adapter_dir / "adapter_model.safetensors") == complete["adapter_model_sha256"]
assert _file_sha256(adapter_dir / "adapter_config.json") == complete["adapter_config_sha256"]

stage = WORK / "account-a-direct-private-stage"
assert not stage.exists(), "Restart the kernel before rebuilding the direct candidate archive"
(stage / "adapter").mkdir(parents=True)
shutil.copy2(complete_path, stage / "complete.json")
shutil.copy2(adapter_dir / "adapter_model.safetensors", stage / "adapter/adapter_model.safetensors")
shutil.copy2(adapter_dir / "adapter_config.json", stage / "adapter/adapter_config.json")
manifest = {}
for path in sorted(stage.rglob("*")):
    if path.is_file():
        relative = path.relative_to(stage).as_posix()
        manifest[relative] = {"bytes": path.stat().st_size, "sha256": _file_sha256(path)}
(stage / "manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")

bundle = WORK / "E45_ACCOUNT_A_CANDIDATE.bin"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED, allowZip64=True) as archive:
    for path in sorted(stage.rglob("*")):
        if path.is_file(): archive.write(path, path.relative_to(stage).as_posix())
bundle_sha = _file_sha256(bundle)
Path(str(bundle) + ".sha256").write_text(f"{bundle_sha}  {bundle.name}\n", encoding="ascii")
print({"direct_private_candidate": str(bundle), "sha256": bundle_sha, "global_step": 705, "holdout_generation": "SKIPPED_BY_USER_AUTHORIZATION"}, flush=True)


In [ ]:
# Final local structural check before Kaggle publishes Output.
import tempfile
from pathlib import Path
import sys
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from uit_dsc_fixed_rag.e45_checkpoint import safe_extract_archive
with tempfile.TemporaryDirectory(prefix="e45_direct_candidate_check_") as temporary:
    extracted = Path(temporary) / "candidate"
    safe_extract_archive(bundle, extracted)
    declared = json.loads((extracted / "manifest.json").read_text(encoding="utf-8"))
    actual = {p.relative_to(extracted).as_posix() for p in extracted.rglob("*") if p.is_file()}
    assert actual == set(declared) | {"manifest.json"}
    for relative, metadata in declared.items():
        path = extracted / relative
        assert path.stat().st_size == metadata["bytes"] and _file_sha256(path) == metadata["sha256"]
print({"status": "READY_FOR_DIRECT_PRIVATE_ADMISSION", "archive": str(bundle), "sidecar": str(Path(str(bundle) + ".sha256"))}, flush=True)
